# LeetCode 1577 — Number of Ways Where Square of Number Is Equal to Product of Two Numbers bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-where-square-of-number-is-equal-to-product-of-two-numbers/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def numTriplets(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu định nghĩa bộ ba **loại 1** và **loại 2**, nhận ra hai loại **đối xứng** ⇒ chỉ cần viết một hàm `count(A, B)` rồi gọi `count(nums1, nums2) + count(nums2, nums1)`.
2. Viết baseline ba vòng lặp `O(n·m²)` và hiểu vì sao chậm.
3. Dùng **`Counter` tích của mọi cặp** `j < k` để trả lời mỗi bình phương trong `O(1)`.
4. Dùng **`Counter` giá trị + duyệt ước số** để đếm cặp có tích bằng `x²` mà không cần liệt kê mọi cặp — xử lý đúng trường hợp **hai số bằng nhau** bằng tổ hợp `C(c, 2)`.
5. Dùng **sắp xếp + hai con trỏ** đếm cặp có tích cố định, xử lý khối giá trị trùng.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên dương `nums1` và `nums2`. Đếm số bộ ba `(i, j, k)` thuộc một trong hai loại:

- **Loại 1:** `nums1[i]² == nums2[j] * nums2[k]` với `0 ≤ i < len(nums1)` và `0 ≤ j < k < len(nums2)`.
- **Loại 2:** `nums2[i]² == nums1[j] * nums1[k]` với `0 ≤ i < len(nums2)` và `0 ≤ j < k < len(nums1)`.

Trả về tổng số bộ ba (loại 1 + loại 2).

**Ví dụ 1:** `nums1 = [7,4]`, `nums2 = [5,2,8,9]` → `1`. Loại 1: `(1,1,2)` vì `4² = 2·8`.

**Ví dụ 2:** `nums1 = [1,1]`, `nums2 = [1,1,1]` → `9`. Loại 1 có `2 × 3 = 6` bộ, loại 2 có `3 × 1 = 3` bộ.

**Ví dụ 3:** `nums1 = [7,7,8,3]`, `nums2 = [1,2,9,7]` → `2`. Loại 1: `(3,0,2)` vì `3² = 1·9`. Loại 2: `(3,0,1)` vì `7² = 7·7`.

**Ràng buộc:**
- `1 ≤ nums1.length, nums2.length ≤ 1000`
- `1 ≤ nums1[i], nums2[i] ≤ 10⁵`

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [([7, 4], [5, 2, 8, 9], 1), ([1, 1], [1, 1, 1], 9), ([7, 7, 8, 3], [1, 2, 9, 7], 2)]
for a, b, want in examples:
    print(f"nums1={a}, nums2={b} -> mong đợi {want}")

nums1=[7, 4], nums2=[5, 2, 8, 9] -> mong đợi 1
nums1=[1, 1], nums2=[1, 1, 1] -> mong đợi 9
nums1=[7, 7, 8, 3], nums2=[1, 2, 9, 7] -> mong đợi 2


## 3. Kiến thức nền

### 3.1 Duyệt mọi cặp `j < k`

Hai vòng lồng nhau `for j in range(m): for k in range(j+1, m)` liệt kê mỗi cặp **không thứ tự** đúng một lần. `itertools.combinations(arr, 2)` làm việc tương tự.

In [2]:
from itertools import combinations
arr = [5, 2, 8]
print([(j, k) for j in range(len(arr)) for k in range(j + 1, len(arr))])
print(list(combinations(arr, 2)))
print("số cặp với m phần tử = m(m-1)/2, m=3 ->", 3 * 2 // 2)

[(0, 1), (0, 2), (1, 2)]
[(5, 2), (5, 8), (2, 8)]
số cặp với m phần tử = m(m-1)/2, m=3 -> 3


### 3.2 `collections.Counter`

`Counter(iterable)` đếm số lần xuất hiện của mỗi giá trị. Truy cập khoá không tồn tại trả `0` (không lỗi). Có thể cộng dồn bằng `cnt[key] += 1`.

In [3]:
cnt = Counter([7, 7, 8, 3])
print(cnt, "| cnt[7] =", cnt[7], "| cnt[100] =", cnt[100])
prod_cnt = Counter(a * b for a, b in combinations([1, 2, 9, 7], 2))
print("tích các cặp của [1,2,9,7]:", prod_cnt)

Counter({7: 2, 8: 1, 3: 1}) | cnt[7] = 2 | cnt[100] = 0
tích các cặp của [1,2,9,7]: Counter({2: 1, 9: 1, 7: 1, 18: 1, 14: 1, 63: 1})


### 3.3 Tổ hợp chập 2 khi có giá trị trùng

Nếu giá trị `a` xuất hiện `c` lần thì số cặp `j < k` mà cả hai đều bằng `a` là `C(c, 2) = c(c-1)/2`. Nếu `a ≠ b`, số cặp (một bằng `a`, một bằng `b`) là `c_a · c_b`.

In [4]:
c = Counter([3, 3, 3, 3, 1, 9])
print("cặp (3,3):", c[3] * (c[3] - 1) // 2, "| cặp (1,9):", c[1] * c[9])
# kiểm tra bằng liệt kê: các cặp có tích 9
print("liệt kê:", sum(1 for x, y in combinations([3, 3, 3, 3, 1, 9], 2) if x * y == 9))

cặp (3,3): 6 | cặp (1,9): 1
liệt kê: 7


### 3.4 Ước số và phép chia hết

`a * b == t` với số nguyên dương ⇔ `t % a == 0` và `b == t // a`. Với `t = x²`, nếu `a < x` thì `b = t/a > x`; nếu `a = x` thì `b = x`. Do đó ta có thể chỉ xét `a ≤ x` và tự suy ra `b ≥ x` — tránh đếm hai lần.

In [5]:
x = 6
t = x * x
pairs = [(a, t // a) for a in range(1, x + 1) if t % a == 0]
print(f"các cặp (a ≤ b) có a*b = {t}:", pairs)

các cặp (a ≤ b) có a*b = 36: [(1, 36), (2, 18), (3, 12), (4, 9), (6, 6)]


### 3.5 Sắp xếp + hai con trỏ đếm cặp có tích bằng `t`

Trên mảng đã sắp tăng (số dương), đặt `l = 0`, `r = m-1`: nếu `arr[l]*arr[r] < t` thì tăng `l`; nếu `> t` thì giảm `r`; nếu bằng thì đếm (cẩn thận khối trùng).

In [6]:
arr = sorted([1, 2, 3, 4, 6, 12])
t, l, r, found = 12, 0, len(arr) - 1, []
while l < r:
    p = arr[l] * arr[r]
    if p < t:
        l += 1
    elif p > t:
        r -= 1
    else:
        found.append((arr[l], arr[r]))
        l += 1
        r -= 1
print("cặp tích 12 (mảng không trùng):", found)

cặp tích 12 (mảng không trùng): [(1, 12), (2, 6), (3, 4)]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `nums1 = [7, 7, 8, 3]`, `nums2 = [1, 2, 9, 7]`.

**Loại 1** — bình phương của `nums1`: `49, 49, 64, 9`. Tích các cặp của `nums2`:

| cặp (j,k) | (0,1) | (0,2) | (0,3) | (1,2) | (1,3) | (2,3) |
|---|---|---|---|---|---|---|
| tích | 2 | **9** | 7 | 18 | 14 | 63 |

Chỉ `9 = 3²` khớp ⇒ 1 bộ (i = 3).

**Loại 2** — bình phương của `nums2`: `1, 4, 81, 49`. Tích các cặp của `nums1`: `(7,7)→49`, `(7,8)→56` ×2, `(7,3)→21` ×2, `(8,3)→24`. `49 = 7²` khớp ⇒ 1 bộ.

Tổng = 2. Nhận xét: loại 2 chính là loại 1 khi **đổi vai** hai mảng ⇒ viết `count(A, B)` = số bộ `(i, j<k)` với `A[i]² = B[j]·B[k]`.

In [7]:
nums1, nums2 = [7, 7, 8, 3], [1, 2, 9, 7]
prods2 = {(j, k): nums2[j] * nums2[k] for j in range(4) for k in range(j + 1, 4)}
print("tích cặp nums2:", prods2)
print("bình phương nums1:", [x * x for x in nums1])
print("loại 1 khớp:", [(i, j, k) for i, x in enumerate(nums1) for (j, k), p in prods2.items() if x * x == p])

tích cặp nums2: {(0, 1): 2, (0, 2): 9, (0, 3): 7, (1, 2): 18, (1, 3): 14, (2, 3): 63}
bình phương nums1: [49, 49, 64, 9]
loại 1 khớp: [(3, 0, 2)]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đúng định nghĩa — với mỗi `i` trong `A`, duyệt mọi cặp `j < k` trong `B`.

In [8]:
def count_brute(A, B):
    total = 0
    for x in A:
        sq = x * x
        for j in range(len(B)):
            for k in range(j + 1, len(B)):
                if B[j] * B[k] == sq:
                    total += 1
    return total

def brute_force(nums1, nums2):
    return count_brute(nums1, nums2) + count_brute(nums2, nums1)

for a, b, want in examples:
    print(a, b, "->", brute_force(a, b), "| mong đợi", want)

[7, 4] [5, 2, 8, 9] -> 1 | mong đợi 1
[1, 1] [1, 1, 1] -> 9 | mong đợi 9
[7, 7, 8, 3] [1, 2, 9, 7] -> 2 | mong đợi 2


**Độ phức tạp:** `O(n·m² + m·n²)` với `n = len(nums1)`, `m = len(nums2)`. Với `n = m = 1000` là khoảng `10⁹` phép nhân — quá chậm.

**Nút thắt:** tích các cặp của `B` được **tính lại** cho mỗi `x` trong `A`. Tính một lần và lưu lại sẽ tiết kiệm được hệ số `n`.

In [9]:
rng = random.Random(1)
a = [rng.randint(1, 20) for _ in range(120)]
b = [rng.randint(1, 20) for _ in range(120)]
t0 = time.perf_counter()
r = brute_force(a, b)
print(f"n=m=120: {r}, mất {time.perf_counter() - t0:.3f}s (n=m=1000 sẽ chậm gấp ~580 lần)")

n=m=120: 8493, mất 0.070s (n=m=1000 sẽ chậm gấp ~580 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đối xứng:** `đáp án = count(nums1, nums2) + count(nums2, nums1)`.

**Insight 2 — tách hai phía:** điều kiện `A[i]² = B[j]·B[k]` có dạng "giá trị phía trái = giá trị phía phải". Nếu ta biết `P[t]` = số cặp `j<k` của `B` có tích `t`, thì `count(A, B) = Σ_i P[A[i]²]`. Đây là pattern **"đếm bằng bảng băm"** (như Two Sum đếm).

**Insight 3 — không cần liệt kê cặp:** với `t = x²` cố định, cặp `(a, b)` với `a·b = t` và `a ≤ b`:
- nếu `a < b` (tức `a < x`): đóng góp `cnt[a] · cnt[b]`;
- nếu `a = b = x`: đóng góp `C(cnt[x], 2)`.

Duyệt `a` qua các **giá trị phân biệt** của `B` (thay vì mọi cặp) ⇒ `O(d_A · d_B)` với `d` = số giá trị phân biệt.

**Insight 4 — tích có thể lớn:** `10⁵ · 10⁵ = 10¹⁰` — Python không tràn số, nhưng ở Java/C++ phải dùng `long`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` tích của mọi cặp

**Bước 1:** `P = Counter(B[j]*B[k] for j<k)` — `O(m²)` thời gian và bộ nhớ.

**Bước 2:** `count(A, B) = Σ P[x*x] for x in A` — `O(n)`.

Gọi hai lần với vai trò đổi nhau.

In [10]:
def count_pair_products(A, B):
    P = Counter()
    m = len(B)
    for j in range(m):
        bj = B[j]
        for k in range(j + 1, m):
            P[bj * B[k]] += 1
    return sum(P[x * x] for x in A)

def solve_pair_products(nums1, nums2):
    return count_pair_products(nums1, nums2) + count_pair_products(nums2, nums1)

for a, b, want in examples:
    print(a, b, "->", solve_pair_products(a, b), "| mong đợi", want)

[7, 4] [5, 2, 8, 9] -> 1 | mong đợi 1
[1, 1] [1, 1, 1] -> 9 | mong đợi 9
[7, 7, 8, 3] [1, 2, 9, 7] -> 2 | mong đợi 2


Code nộp LeetCode cho cách 2:

```python
class Solution:
    def numTriplets(self, nums1: List[int], nums2: List[int]) -> int:
        def count(A, B):
            P = Counter()
            for j in range(len(B)):
                for k in range(j + 1, len(B)):
                    P[B[j] * B[k]] += 1
            return sum(P[x * x] for x in A)
        return count(nums1, nums2) + count(nums2, nums1)
```

**Độ phức tạp:** `O(n² + m²)` thời gian, `O(n² + m²)` bộ nhớ (tệ nhất ~500 000 khoá mỗi bảng). Đủ nhanh cho `1000`, nhưng tốn bộ nhớ.

### 7.2 Cách 3 — `Counter` giá trị + duyệt từng bình phương (lời giải gọn và nhanh)

**Bước 1:** `cA = Counter(A)`, `cB = Counter(B)`.

**Bước 2:** với mỗi giá trị phân biệt `x` (xuất hiện `cA[x]` lần) trong `A`, đặt `t = x²`. Duyệt mỗi giá trị phân biệt `a` trong `B`:
- nếu `t % a != 0` thì bỏ qua;
- `b = t // a`; nếu `a < b`: cộng `cB[a] * cB[b]`; nếu `a == b`: cộng `cB[a]*(cB[a]-1)//2`; nếu `a > b`: bỏ (đã đếm khi duyệt tới `b`).

**Bước 3:** nhân số cặp tìm được với `cA[x]` (mỗi chỉ số `i` có giá trị `x` tạo cùng số bộ ba).

In [11]:
def count_by_value(A, B):
    cA, cB = Counter(A), Counter(B)
    total = 0
    for x, fx in cA.items():
        t = x * x
        pairs = 0
        for a, fa in cB.items():
            if t % a:
                continue
            b = t // a
            if a < b:
                pairs += fa * cB[b]       # cB[b] = 0 nếu b không có
            elif a == b:
                pairs += fa * (fa - 1) // 2
        total += fx * pairs
    return total

def solve_by_value(nums1, nums2):
    return count_by_value(nums1, nums2) + count_by_value(nums2, nums1)

for a, b, want in examples:
    print(a, b, "->", solve_by_value(a, b), "| mong đợi", want)

[7, 4] [5, 2, 8, 9] -> 1 | mong đợi 1
[1, 1] [1, 1, 1] -> 9 | mong đợi 9
[7, 7, 8, 3] [1, 2, 9, 7] -> 2 | mong đợi 2


Code nộp LeetCode cho cách 3 (giống phần 9):

```python
class Solution:
    def numTriplets(self, nums1: List[int], nums2: List[int]) -> int:
        def count(A, B):
            cA, cB = Counter(A), Counter(B)
            total = 0
            for x, fx in cA.items():
                t = x * x
                pairs = 0
                for a, fa in cB.items():
                    if t % a:
                        continue
                    b = t // a
                    if a < b:
                        pairs += fa * cB[b]
                    elif a == b:
                        pairs += fa * (fa - 1) // 2
                total += fx * pairs
            return total
        return count(nums1, nums2) + count(nums2, nums1)
```

**Độ phức tạp:** `O(n + m + d₁·d₂)` thời gian với `d₁, d₂` = số giá trị phân biệt (`≤ 1000`), `O(d₁ + d₂)` bộ nhớ. Nhanh khi có nhiều giá trị trùng.

### 7.3 Cách 4 — Sắp xếp + hai con trỏ cho mỗi bình phương

**Ý tưởng:** sắp `B`. Với mỗi `x` trong `A`, đếm số cặp `l < r` có `B[l]·B[r] = x²` bằng hai con trỏ. Khi tích bằng:
- nếu `B[l] == B[r]`: mọi phần tử trong `[l..r]` bằng nhau ⇒ cộng `C(r-l+1, 2)` rồi dừng;
- ngược lại: đếm khối trùng bên trái (`cl` phần tử bằng `B[l]`) và bên phải (`cr` phần tử bằng `B[r]`) ⇒ cộng `cl · cr`, nhảy qua hai khối.

Không cần bảng băm, bộ nhớ `O(1)` phụ (ngoài bản sắp).

In [12]:
def count_two_pointers(A, B):
    B = sorted(B)
    total = 0
    for x in A:
        t = x * x
        l, r = 0, len(B) - 1
        while l < r:
            p = B[l] * B[r]
            if p < t:
                l += 1
            elif p > t:
                r -= 1
            elif B[l] == B[r]:
                k = r - l + 1
                total += k * (k - 1) // 2
                break
            else:
                cl = 1
                while l + cl < r and B[l + cl] == B[l]:
                    cl += 1
                cr = 1
                while r - cr > l and B[r - cr] == B[r]:
                    cr += 1
                total += cl * cr
                l += cl
                r -= cr
    return total

def solve_two_pointers(nums1, nums2):
    return count_two_pointers(nums1, nums2) + count_two_pointers(nums2, nums1)

for a, b, want in examples:
    print(a, b, "->", solve_two_pointers(a, b), "| mong đợi", want)

[7, 4] [5, 2, 8, 9] -> 1 | mong đợi 1
[1, 1] [1, 1, 1] -> 9 | mong đợi 9
[7, 7, 8, 3] [1, 2, 9, 7] -> 2 | mong đợi 2


Code nộp LeetCode cho cách 4:

```python
class Solution:
    def numTriplets(self, nums1: List[int], nums2: List[int]) -> int:
        def count(A, B):
            B = sorted(B)
            total = 0
            for x in A:
                t, l, r = x * x, 0, len(B) - 1
                while l < r:
                    p = B[l] * B[r]
                    if p < t:
                        l += 1
                    elif p > t:
                        r -= 1
                    elif B[l] == B[r]:
                        k = r - l + 1
                        total += k * (k - 1) // 2
                        break
                    else:
                        cl = cr = 1
                        while l + cl < r and B[l + cl] == B[l]:
                            cl += 1
                        while r - cr > l and B[r - cr] == B[r]:
                            cr += 1
                        total += cl * cr
                        l, r = l + cl, r - cr
            return total
        return count(nums1, nums2) + count(nums2, nums1)
```

**Độ phức tạp:** `O(m log m + n log n + 2·n·m)` thời gian, `O(n + m)` cho bản sắp. Với `n = m = 1000` là ~`2·10⁶` bước — rất ổn.

## 8. TRACE từng bước

Trace cách 3 cho loại 2 của ví dụ 2: `A = nums2 = [1,1,1]`, `B = nums1 = [1,1]`.
- `cA = {1: 3}`, `cB = {1: 2}`.
- `x = 1`, `t = 1`. Duyệt `a = 1`: `t % 1 = 0`, `b = 1 = a` ⇒ `pairs += C(2,2) = 1`.
- `total += cA[1] · pairs = 3 · 1 = 3`.

Loại 1 (`A = [1,1]`, `B = [1,1,1]`): `pairs = C(3,2) = 3`, `total = 2 · 3 = 6`. Tổng `9`. ✔

In [13]:
def trace_count_by_value(A, B, name=""):
    cA, cB = Counter(A), Counter(B)
    print(f"[{name}] cA={dict(cA)}  cB={dict(cB)}")
    total = 0
    for x, fx in cA.items():
        t = x * x
        pairs = 0
        for a, fa in cB.items():
            if t % a:
                print(f"   x={x} t={t}: a={a} không chia hết -> bỏ")
                continue
            b = t // a
            if a < b:
                add = fa * cB[b]
                print(f"   x={x} t={t}: a={a} b={b} (a<b) -> cộng cB[{a}]*cB[{b}] = {fa}*{cB[b]} = {add}")
                pairs += add
            elif a == b:
                add = fa * (fa - 1) // 2
                print(f"   x={x} t={t}: a=b={a} -> cộng C({fa},2) = {add}")
                pairs += add
            else:
                print(f"   x={x} t={t}: a={a} b={b} (a>b) -> bỏ, đã đếm khi a={b}")
        total += fx * pairs
        print(f"  -> x={x}: pairs={pairs}, nhân cA[x]={fx}, total={total}")
    return total

t1 = trace_count_by_value([7, 7, 8, 3], [1, 2, 9, 7], "loại 1")
t2 = trace_count_by_value([1, 2, 9, 7], [7, 7, 8, 3], "loại 2")
print("đáp án =", t1, "+", t2, "=", t1 + t2)

[loại 1] cA={7: 2, 8: 1, 3: 1}  cB={1: 1, 2: 1, 9: 1, 7: 1}
   x=7 t=49: a=1 b=49 (a<b) -> cộng cB[1]*cB[49] = 1*0 = 0
   x=7 t=49: a=2 không chia hết -> bỏ
   x=7 t=49: a=9 không chia hết -> bỏ
   x=7 t=49: a=b=7 -> cộng C(1,2) = 0
  -> x=7: pairs=0, nhân cA[x]=2, total=0
   x=8 t=64: a=1 b=64 (a<b) -> cộng cB[1]*cB[64] = 1*0 = 0
   x=8 t=64: a=2 b=32 (a<b) -> cộng cB[2]*cB[32] = 1*0 = 0
   x=8 t=64: a=9 không chia hết -> bỏ
   x=8 t=64: a=7 không chia hết -> bỏ
  -> x=8: pairs=0, nhân cA[x]=1, total=0
   x=3 t=9: a=1 b=9 (a<b) -> cộng cB[1]*cB[9] = 1*1 = 1
   x=3 t=9: a=2 không chia hết -> bỏ
   x=3 t=9: a=9 b=1 (a>b) -> bỏ, đã đếm khi a=1
   x=3 t=9: a=7 không chia hết -> bỏ
  -> x=3: pairs=1, nhân cA[x]=1, total=1
[loại 2] cA={1: 1, 2: 1, 9: 1, 7: 1}  cB={7: 2, 8: 1, 3: 1}
   x=1 t=1: a=7 không chia hết -> bỏ
   x=1 t=1: a=8 không chia hết -> bỏ
   x=1 t=1: a=3 không chia hết -> bỏ
  -> x=1: pairs=0, nhân cA[x]=1, total=0
   x=2 t=4: a=7 không chia hết -> bỏ
   x=2 t=4: a=8 không c

In [14]:
def trace_two_pointers_one(x, B):
    B = sorted(B)
    t, l, r, total = x * x, 0, len(B) - 1, 0
    print(f"x={x}, t={t}, B sắp = {B}")
    while l < r:
        p = B[l] * B[r]
        if p < t:
            print(f"  l={l} r={r}: {B[l]}*{B[r]}={p} < {t} -> l+=1")
            l += 1
        elif p > t:
            print(f"  l={l} r={r}: {B[l]}*{B[r]}={p} > {t} -> r-=1")
            r -= 1
        elif B[l] == B[r]:
            k = r - l + 1
            total += k * (k - 1) // 2
            print(f"  l={l} r={r}: bằng, cả khối {k} phần tử bằng {B[l]} -> cộng C({k},2), dừng")
            break
        else:
            cl = 1
            while l + cl < r and B[l + cl] == B[l]:
                cl += 1
            cr = 1
            while r - cr > l and B[r - cr] == B[r]:
                cr += 1
            total += cl * cr
            print(f"  l={l} r={r}: bằng, khối trái {cl} x khối phải {cr} -> cộng {cl * cr}")
            l += cl
            r -= cr
    print("  số cặp =", total)
    return total

trace_two_pointers_one(6, [2, 2, 3, 6, 6, 12, 18, 18, 4, 9])

x=6, t=36, B sắp = [2, 2, 3, 4, 6, 6, 9, 12, 18, 18]
  l=0 r=9: bằng, khối trái 2 x khối phải 2 -> cộng 4
  l=2 r=7: bằng, khối trái 1 x khối phải 1 -> cộng 1
  l=3 r=6: bằng, khối trái 1 x khối phải 1 -> cộng 1
  l=4 r=5: bằng, cả khối 2 phần tử bằng 6 -> cộng C(2,2), dừng
  số cặp = 7


7

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def numTriplets(self, nums1: List[int], nums2: List[int]) -> int:
        def count(A, B):
            cA, cB = Counter(A), Counter(B)
            total = 0
            for x, fx in cA.items():
                t = x * x
                pairs = 0
                for a, fa in cB.items():
                    if t % a:
                        continue
                    b = t // a
                    if a < b:
                        pairs += fa * cB[b]
                    elif a == b:
                        pairs += fa * (fa - 1) // 2
                total += fx * pairs
            return total

        return count(nums1, nums2) + count(nums2, nums1)

sol = Solution()
for a, b, want in examples:
    print(a, b, "->", sol.numTriplets(a, b), "| mong đợi", want)

[7, 4] [5, 2, 8, 9] -> 1 | mong đợi 1
[1, 1] [1, 1, 1] -> 9 | mong đợi 9
[7, 7, 8, 3] [1, 2, 9, 7] -> 2 | mong đợi 2


## 10. Vì sao thuật toán đúng

Xét `count(A, B)` = số bộ `(i, {j, k})` với `j < k` và `A[i]² = B[j]·B[k]`.

**Cách 2:** `P[t]` đếm chính xác số cặp `j < k` có tích `t` (mỗi cặp được liệt kê đúng một lần). Tổng `Σ_i P[A[i]²]` đếm mọi bộ theo định nghĩa.

**Cách 3:** Cố định `x = A[i]`, `t = x²`. Mỗi cặp chỉ số `j < k` có tích `t` ứng với cặp giá trị `{a, b}` (không thứ tự) có `a·b = t`. Phân loại theo cặp giá trị với `a ≤ b`:
- `a < b`: số cặp chỉ số = (số vị trí bằng `a`) × (số vị trí bằng `b`) = `cB[a]·cB[b]` (mỗi cặp vị trí khác nhau được sắp thành `j<k` đúng một cách).
- `a = b`: chọn 2 vị trí phân biệt trong `cB[a]` vị trí ⇒ `C(cB[a], 2)`.
Mỗi cặp giá trị `{a, b}` được đếm đúng một lần vì ta chỉ cộng khi `a ≤ b` (với `a > b` thì bỏ qua). Tất cả chỉ số `i` có cùng giá trị `x` cho cùng số cặp ⇒ nhân `cA[x]`.

**Cách 4:** Hai con trỏ trên mảng sắp tăng gồm số dương: nếu `B[l]·B[r] < t` thì `B[l]` nhân với **bất kỳ** phần tử nào trong `(l, r]` cũng `≤ B[l]·B[r] < t` ⇒ bỏ `l` an toàn; đối xứng cho `> t`. Khi bằng, đếm toàn bộ tổ hợp giữa hai khối trùng rồi bỏ cả hai khối (mọi phần tử trong khối trái chỉ có thể ghép với giá trị `B[r]`).

## 11. Phân tích Time / Space

Gọi `n = len(nums1)`, `m = len(nums2)`, `d₁, d₂` = số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n·m² + m·n²)` | `O(1)` |
| 2. Counter tích cặp | `O(n² + m²)` | `O(n² + m²)` |
| 3. Counter giá trị | `O(n + m + d₁·d₂)` | `O(d₁ + d₂)` |
| 4. Sắp + hai con trỏ | `O(n log n + m log m + n·m)` | `O(n + m)` |

Với ràng buộc `≤ 1000`, cách 3 và 4 tốn cỡ `10⁶` bước; cách 2 cỡ `10⁶` thao tác băm và tối đa ~`5·10⁵` khoá mỗi bảng.

## 12. Bộ test

In [16]:
ALL = [brute_force, solve_pair_products, solve_by_value, solve_two_pointers, Solution().numTriplets]

for f in ALL:
    for a, b, want in examples:
        assert f(list(a), list(b)) == want, (f.__name__, a, b)
print("Ví dụ đề: OK")

edge = [
    ([1], [1], 0),                    # mỗi mảng 1 phần tử: không có cặp j<k
    ([2], [4, 1], 1),                 # 2^2 = 4*1
    ([3], [3, 3, 3, 3], 6),           # C(4,2) = 6 cặp (3,3)
    ([100000], [100000, 100000], 1),  # giá trị lớn nhất, tích 10^10
    ([4, 4], [2, 8, 16, 1], 4),       # 16 = 2*8 = 16*1, nhân 2 chỉ số i
    ([5, 5, 5], [5, 5, 5], 18),       # mỗi loại: 3 * C(3,2) = 9
]
for f in ALL:
    for a, b, want in edge:
        assert f(list(a), list(b)) == want, (f.__name__, a, b, f(list(a), list(b)), want)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
# Ngẫu nhiên seed cố định, giá trị nhỏ để có nhiều trùng/khớp
rng = random.Random(1577)
for _ in range(1500):
    n, m = rng.randint(1, 12), rng.randint(1, 12)
    pool = rng.choice([[1, 2, 4], [1, 2, 3, 4, 6, 8, 9, 12, 16, 18], list(range(1, 30))])
    a = [rng.choice(pool) for _ in range(n)]
    b = [rng.choice(pool) for _ in range(m)]
    want = brute_force(a, b)
    for f in ALL[1:]:
        assert f(list(a), list(b)) == want, (f.__name__, a, b)
print("1500 cặp mảng ngẫu nhiên: OK")

1500 cặp mảng ngẫu nhiên: OK


In [18]:
# Test lớn đúng ràng buộc: n = m = 1000
rng = random.Random(42)
big_tests = [
    ([rng.randint(1, 10**5) for _ in range(1000)], [rng.randint(1, 10**5) for _ in range(1000)]),
    ([rng.choice([1, 2, 4, 8, 16]) for _ in range(1000)], [rng.choice([1, 2, 4, 8, 16]) for _ in range(1000)]),
    ([1] * 1000, [1] * 1000),
]
for a, b in big_tests:
    res = []
    for f in [solve_pair_products, solve_by_value, solve_two_pointers]:
        t0 = time.perf_counter()
        res.append(f(a, b))
        print(f"  {f.__name__:<20} {res[-1]:>12}  {time.perf_counter() - t0:.3f}s")
    assert len(set(res)) == 1
assert solve_by_value([1] * 1000, [1] * 1000) == 2 * 1000 * (1000 * 999 // 2)
print("Test lớn: OK")

  solve_pair_products             1  0.379s
  solve_by_value                  1  0.076s


  solve_two_pointers              1  0.154s
  solve_pair_products     103889196  0.136s
  solve_by_value          103889196  0.000s


  solve_two_pointers      103889196  0.102s
  solve_pair_products     999000000  0.124s
  solve_by_value          999000000  0.000s
  solve_two_pointers      999000000  0.000s
Test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `cnt[a] * cnt[a]` khi `a == b`

Cặp `j < k` cần **hai vị trí khác nhau**, nên phải là `C(c, 2)`, không phải `c²` (đếm cả `j = k` và cả hai thứ tự).

In [19]:
def wrong_same_value(A, B):
    cA, cB = Counter(A), Counter(B)
    total = 0
    for x, fx in cA.items():
        t = x * x
        for a, fa in cB.items():
            if t % a == 0:
                b = t // a
                if a < b:
                    total += fx * fa * cB[b]
                elif a == b:
                    total += fx * fa * fa          # SAI: phải là fa*(fa-1)//2
    return total

a, b = [1, 1], [1, 1, 1]
print("sai:", wrong_same_value(a, b) + wrong_same_value(b, a), "| đúng:", Solution().numTriplets(a, b))

sai: 30 | đúng: 9


### Lỗi 2 — Đếm cặp **có thứ tự** (cả `(a, b)` và `(b, a)`)

Nếu cộng `cB[a]·cB[b]` cho mọi `a` (kể cả `a > b`) thì mỗi cặp bị đếm hai lần.

In [20]:
def wrong_ordered(A, B):
    cA, cB = Counter(A), Counter(B)
    total = 0
    for x, fx in cA.items():
        t = x * x
        for a, fa in cB.items():
            if t % a == 0:
                b = t // a
                if a != b:
                    total += fx * fa * cB[b]       # SAI: thiếu điều kiện a < b
                else:
                    total += fx * fa * (fa - 1) // 2
    return total

a, b = [7, 4], [5, 2, 8, 9]
print("sai:", wrong_ordered(a, b) + wrong_ordered(b, a), "| đúng:", Solution().numTriplets(a, b))

sai: 2 | đúng: 1


### Lỗi 3 — Quên loại 2 (chỉ đếm một chiều)

In [21]:
def wrong_one_direction(nums1, nums2):
    return count_by_value(nums1, nums2)            # SAI: thiếu count_by_value(nums2, nums1)

a, b = [7, 7, 8, 3], [1, 2, 9, 7]
print("sai:", wrong_one_direction(a, b), "| đúng:", Solution().numTriplets(a, b))

sai: 1 | đúng: 2


### Lỗi 4 — Chia nguyên mà không kiểm tra chia hết

`b = t // a` luôn ra một số nguyên, kể cả khi `a` không phải ước của `t` ⇒ đếm nhầm những cặp `a·b ≠ t`.

In [22]:
def wrong_no_divisibility(A, B):
    cA, cB = Counter(A), Counter(B)
    total = 0
    for x, fx in cA.items():
        t = x * x
        for a, fa in cB.items():
            b = t // a                              # SAI: thiếu if t % a == 0
            if a < b:
                total += fx * fa * cB[b]
            elif a == b:
                total += fx * fa * (fa - 1) // 2
    return total

a, b = [3], [2, 4]          # 9 // 2 = 4 -> tưởng 2*4 = 9
print("sai:", wrong_no_divisibility(a, b) + wrong_no_divisibility(b, a), "| đúng:", Solution().numTriplets(a, b))

sai: 1 | đúng: 0


### Lỗi 5 — Hai con trỏ: khi tích bằng chỉ tăng `l` một bước, bỏ qua khối trùng

Với `B = [2, 2, 8, 8]` và `t = 16` có `2 × 2 = 4` cặp, nhưng tăng/giảm một bước chỉ đếm 2.

In [23]:
def wrong_two_pointers(A, B):
    B = sorted(B)
    total = 0
    for x in A:
        t, l, r = x * x, 0, len(B) - 1
        while l < r:
            p = B[l] * B[r]
            if p < t:
                l += 1
            elif p > t:
                r -= 1
            else:
                total += 1                          # SAI: bỏ qua tổ hợp giữa các khối trùng
                l += 1
                r -= 1
    return total

print("sai:", wrong_two_pointers([4], [2, 2, 8, 8]), "| đúng:", count_two_pointers([4], [2, 2, 8, 8]))

sai: 2 | đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Hai loại đối xứng** ⇒ viết một hàm phụ, gọi hai lần với tham số đổi chỗ. Giảm một nửa số dòng code và lỗi.
- **"Đếm bộ thoả `f(trái) = g(phải)`"** ⇒ đếm giá trị một phía vào `Counter`, rồi tra phía còn lại (giống 2-Sum đếm, 4-Sum II).
- Đếm cặp không thứ tự có tích/tổng bằng `t` từ `Counter`: chỉ xét `a ≤ b`, tách riêng `a == b` dùng `C(c, 2)`.
- Mảng tối đa `1000` ⇒ `O(n²)` (`10⁶`) là chấp nhận được; `O(n³)` thì không.
- Kiểm tra nhanh bằng ví dụ toàn số 1: `nums1 = [1]*n`, `nums2 = [1]*m` ⇒ `n·C(m,2) + m·C(n,2)`.

## 15. Biến thể của bài

1. **LC 1726 — Tuple with Same Product:** đếm bộ `(a, b, c, d)` phân biệt có `a·b = c·d` ⇒ Counter tích cặp, mỗi tích xuất hiện `f` lần cho `8·C(f, 2)` bộ (bài tập 5).
2. **LC 454 — 4Sum II:** đếm `(i,j,k,l)` với `A[i]+B[j]+C[k]+D[l] = 0` ⇒ Counter tổng cặp của (A,B), tra (C,D).
3. **LC 1010 — Pairs of Songs With Total Durations Divisible by 60:** đếm cặp theo phần dư.
4. **Tổng thay vì tích:** `2·nums1[i] = nums2[j] + nums2[k]` ⇒ cùng khung, đổi điều kiện ghép `b = t - a`.
5. **Một mảng duy nhất:** đếm `(i, j<k)` trong cùng mảng (có thể trùng chỉ số với `i`) — chú ý xem đề có yêu cầu `i ∉ {j, k}` hay không.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | ba vòng lặp theo định nghĩa | `O(n·m² + m·n²)` | `O(1)` | kiểm thử, mảng rất nhỏ |
| 2. Counter tích cặp | tính trước tích mọi cặp, tra bình phương | `O(n² + m²)` | `O(n² + m²)` | dễ nghĩ nhất sau brute force |
| 3. Counter giá trị | với mỗi `x²`, duyệt ước `a` trong giá trị phân biệt | `O(n + m + d₁·d₂)` | `O(d₁ + d₂)` | gọn, nhanh, ít bộ nhớ |
| 4. Sắp + hai con trỏ | đếm cặp có tích `x²` trên mảng sắp | `O(n·m + sort)` | `O(n + m)` | không dùng hash, minh hoạ hai con trỏ với khối trùng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().numTriplets([2, 4], [1, 4, 16, 8])` trả về bao nhiêu?

**Bài 2 (tính tay):** Có bao nhiêu cặp `j < k` trong `[3, 3, 3, 3, 1, 9]` có tích bằng `9`?

**Bài 3 (điền TODO):** Viết `ex3_pairs_with_product(cnt, t)` nhận `Counter` các giá trị và số `t`, trả về số cặp chỉ số `j < k` có tích `t`.

**Bài 4 (tự viết lại):** Viết `ex4_list_type1(nums1, nums2)` trả về **danh sách mọi bộ ba loại 1** `(i, j, k)` (đã sắp), dùng `dict` ánh xạ tích → danh sách cặp `(j, k)`.

**Bài 5 (mini challenge — LC 1726):** Cho `nums` gồm các số nguyên dương **phân biệt**, đếm số bộ `(a, b, c, d)` (các phần tử phân biệt lấy từ `nums`, có thứ tự) sao cho `a·b = c·d`.

In [24]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_pairs_with_product(cnt, t):
    pairs = 0
    for a, fa in cnt.items():
        # TODO: bỏ qua nếu a không chia hết t
        # TODO: b = t // a; a < b -> cộng fa * cnt[b]; a == b -> cộng C(fa, 2)
        pass
    return pairs

In [25]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex4_list_type1(nums1, nums2):
    by_product = {}
    # TODO: với mỗi j < k, thêm (j, k) vào by_product[nums2[j] * nums2[k]]
    res = []
    # TODO: với mỗi i, nối các (i, j, k) từ by_product.get(nums1[i]**2, [])
    return sorted(res)

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Loại 1: `4 = 1·4`, `16 = 1·16 = 4·4 (?)`. (2) Chỉ có một số 4 trong `nums2` nên không có cặp `(4,4)`. (3) Loại 2: bình phương của `nums2` là `1, 16, 256, 64`; tích cặp `nums1` là `8`.

**Bài 2.** (1) Cặp `(3,3)`: `C(4,2)`. (2) Cặp `(1,9)`: `1·1`. (3) Cộng lại.

**Bài 3.** (1) Kiểm tra `t % a`. (2) Chỉ đếm khi `a ≤ b`. (3) `a == b` dùng `fa*(fa-1)//2`.

**Bài 4.** (1) `by_product.setdefault(p, []).append((j, k))`. (2) Duyệt `i`, tra `nums1[i]**2`. (3) Sắp kết quả để so sánh với brute force.

**Bài 5.** (1) Đếm tần suất mỗi tích cặp `nums[i]·nums[j]` (`i<j`). (2) Hai cặp khác nhau cùng tích dùng **4 phần tử phân biệt** (vì các số phân biệt). (3) Mỗi cách chọn 2 cặp cho 8 bộ có thứ tự ⇒ `Σ 8·C(f, 2)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = Solution().numTriplets([2, 4], [1, 4, 16, 8])
print("Bài 1:", ans1)
assert ans1 == brute_force([2, 4], [1, 4, 16, 8]) == 2      # (0: 4=1*4), (1: 16=1*16)

# Bài 2
arr = [3, 3, 3, 3, 1, 9]
ans2 = sum(1 for j in range(len(arr)) for k in range(j + 1, len(arr)) if arr[j] * arr[k] == 9)
print("Bài 2:", ans2)
assert ans2 == 6 + 1
print("Bài 1, 2: OK")

Bài 1: 2
Bài 2: 7
Bài 1, 2: OK


In [27]:
# Bài 3
def ex3_pairs_with_product_sol(cnt, t):
    pairs = 0
    for a, fa in cnt.items():
        if t % a:
            continue
        b = t // a
        if a < b:
            pairs += fa * cnt[b]
        elif a == b:
            pairs += fa * (fa - 1) // 2
    return pairs

rng = random.Random(3)
for _ in range(800):
    B = [rng.randint(1, 12) for _ in range(rng.randint(1, 10))]
    t = rng.randint(1, 60)
    want = sum(1 for j in range(len(B)) for k in range(j + 1, len(B)) if B[j] * B[k] == t)
    assert ex3_pairs_with_product_sol(Counter(B), t) == want
print("Bài 3: OK, ví dụ:", ex3_pairs_with_product_sol(Counter([3, 3, 3, 3, 1, 9]), 9))

Bài 3: OK, ví dụ: 7


In [28]:
# Bài 4
def ex4_list_type1_sol(nums1, nums2):
    by_product = {}
    for j in range(len(nums2)):
        for k in range(j + 1, len(nums2)):
            by_product.setdefault(nums2[j] * nums2[k], []).append((j, k))
    res = [(i, j, k) for i, x in enumerate(nums1) for j, k in by_product.get(x * x, [])]
    return sorted(res)

def ex4_brute(nums1, nums2):
    return sorted((i, j, k) for i in range(len(nums1)) for j in range(len(nums2))
                  for k in range(j + 1, len(nums2)) if nums1[i] ** 2 == nums2[j] * nums2[k])

print("Bài 4:", ex4_list_type1_sol([1, 1], [1, 1, 1]))
assert ex4_list_type1_sol([1, 1], [1, 1, 1]) == [(0, 0, 1), (0, 0, 2), (0, 1, 2), (1, 0, 1), (1, 0, 2), (1, 1, 2)]
rng = random.Random(4)
for _ in range(500):
    a = [rng.randint(1, 8) for _ in range(rng.randint(1, 7))]
    b = [rng.randint(1, 8) for _ in range(rng.randint(1, 7))]
    assert ex4_list_type1_sol(a, b) == ex4_brute(a, b)
print("Bài 4: OK")

Bài 4: [(0, 0, 1), (0, 0, 2), (0, 1, 2), (1, 0, 1), (1, 0, 2), (1, 1, 2)]
Bài 4: OK


In [29]:
# Bài 5 — LC 1726
def ex5_tuple_same_product(nums):
    P = Counter(nums[i] * nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))
    return sum(8 * f * (f - 1) // 2 for f in P.values())

def ex5_brute(nums):
    from itertools import permutations
    return sum(1 for a, b, c, d in permutations(nums, 4) if a * b == c * d)

print("Bài 5:", ex5_tuple_same_product([2, 3, 4, 6]), ex5_tuple_same_product([1, 2, 4, 5, 10]))
assert ex5_tuple_same_product([2, 3, 4, 6]) == 8 and ex5_tuple_same_product([1, 2, 4, 5, 10]) == 16
rng = random.Random(5)
for _ in range(150):
    nums = rng.sample(range(1, 25), rng.randint(1, 7))
    assert ex5_tuple_same_product(nums) == ex5_brute(nums), nums
print("Bài 5: OK")

Bài 5: 8 16
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def numTriplets(self, nums1, nums2):
        def count(A, B):                       # số (i, j<k) với A[i]^2 = B[j]*B[k]
            cA, cB = Counter(A), Counter(B)
            total = 0
            for x, fx in cA.items():
                t, pairs = x * x, 0
                for a, fa in cB.items():
                    if t % a:
                        continue
                    b = t // a
                    if a < b:
                        pairs += fa * cB[b]
                    elif a == b:
                        pairs += fa * (fa - 1) // 2
                total += fx * pairs
            return total
        return count(nums1, nums2) + count(nums2, nums1)
```

**Template "đếm cặp không thứ tự có tích (tổng) = t từ Counter":** duyệt `a`, suy ra `b`; `a < b` → `c[a]·c[b]`; `a == b` → `C(c[a], 2)`; `a > b` → bỏ.

**Checklist trước khi submit:**
- [ ] Cộng **cả hai loại** (gọi hàm phụ hai lần, đổi vai).
- [ ] Kiểm tra `t % a == 0` trước khi lấy `b = t // a`.
- [ ] Chỉ đếm khi `a ≤ b`; `a == b` dùng `C(c, 2)`.
- [ ] Nhân với số lần xuất hiện của `x` trong mảng còn lại.
- [ ] (Java/C++) dùng `long` cho `x*x` và tích — tối đa `10¹⁰`.
- [ ] Thử nhanh: ví dụ → `1, 9, 2`; `[3]`,`[3,3,3,3]` → `6`; `[1]`,`[1]` → `0`.